In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning

from input import input
import config
from model import generics, hybrid_system_exp, grid_search_exp
from model.feature_selection import TimeSeriesFeatureSelector
from metrics import metrics
import numpy as np

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
import pandas as pd

%load_ext autoreload
%autoreload 2

In [ ]:
# Tarefa 3 do PLANO_ARQUITETURA.md: integracao do TimeSeriesFeatureSelector
# via sklearn.Pipeline. strategy fixa nesta execucao ('lasso' -- embedded,
# regularizacao L1 via LassoCV(cv=TimeSeriesSplit(...)) -- cv cronologico
# obrigatorio, nunca KFold aleatorio, ver PLANO_ARQUITETURA.md Secao 2 e
# CLAUDE.md Secao 5.2). k varia como grid. A transformacao de y
# (MinMaxScaler/KPSS) continua fora do Pipeline, dentro de
# Additive/generics.format_forecats (Secao 1.4 do PLANO_ARQUITETURA.md).
model = Pipeline([
    ('selector', TimeSeriesFeatureSelector(strategy='lasso')),
    ('estimator', MLPRegressor(activation='logistic', solver='lbfgs')),
])

# experiment_id novo e explicito (Secao 3.2 do CLAUDE.md).
experiment_id = 'chamados_v3_fs_lasso'
# Sufixo sem underscore -- extract_series_name_from_path/model_name em
# metrics.py continuam funcionando sem nenhuma mudanca (ver Tarefa 3, Parte B).
model_name = 'amv1lasso'
normalize = True
force = False
model_exec = 10

experiment_params = {
    'linear_model_name': '1arima',
    'diff_kpss': False,
    'horizon': 1
}

# Convencao selector__*/estimator__* (nativa do sklearn): GridSearch usa
# clone(model).set_params(**params), que resolve essas chaves automaticamente
# via Pipeline.get_params(deep=True) -- nenhuma mudanca em grid_search_exp.py.
model_parameters = {
    'selector__k': [5, 10, 15],
    'estimator__hidden_layer_sizes': [10, 20, 50],
    'estimator__max_iter': [1000],
}


In [ ]:
# Sanity-check exigido pela Tarefa 2: confirma que Pipeline.get_params(deep=True)
# expoe as chaves selector__*/estimator__* que GridSearch (ParameterGrid +
# clone().set_params()) vai usar, antes de rodar o grid completo.
params = model.get_params(deep=True)
required_keys = {'selector__strategy', 'selector__k', 'estimator__hidden_layer_sizes', 'estimator__max_iter'}
missing = required_keys - params.keys()
assert not missing, f"get_params(deep=True) nao expos chaves esperadas: {missing}"
print("OK -- Pipeline.get_params(deep=True) expoe:", sorted(required_keys))

In [ ]:
# Cobre as 4 series de FS_DEV_SERIES. austres.txt/coloradoRiver.txt exigem
# fs_lag_size em config.py -- confirmado no pre-check da Tarefa 3
# (austres=12, coloradoRiver=30). Chamado direto via GridSearch(...).execution()
# por serie, sem depender/mutar config.BASE_NAME_LIST e sem tocar grid_search_exp.py.
#
# use_val_slipt_for_prev=True e explicito porque o default de GridSearch (False)
# diverge do default do wrapper grid_seach_multiple_bases (True) -- sem isso,
# o refit final perderia o val_size e os .pkl ficariam sem val_metrics.
fs_series_list = ['airlines.txt', 'austres.txt', 'coloradoRiver.txt', 'sunspot.txt']

for base_name in fs_series_list:
    print(base_name)
    exec_gs = grid_search_exp.GridSearch(
        hybrid_system_exp.Additive,
        model,
        model_parameters,
        experiment_id,
        base_name,
        model_name,
        force,
        normalize,
        experiment_params,
        model_exec=model_exec,
        use_val_slipt_for_prev=True,
    )
    exec_gs.execution()